# Machine Learning Hackathon

## Problem Statement  

The objective of this problem is to develop a Machine Learning regression model that can predict the **overall yield of a chemical process** based on its operating conditions.

The dataset contains process parameters such as **flow rate, reactant concentration, inlet and jacket temperatures, and reactor length**. In addition to these raw variables, domain-inspired features such as **residence time, thermal exposure, inverse temperature, and kinetic exposure** are used to capture important relationships within the process.

The goal is to build a model that can accurately predict `overall_yield` for unseen process conditions. Model performance is evaluated using **Root Mean Squared Error (RMSE), where a lower RMSE indicates better predictive performance.


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('train_dataset.csv')  # given data set

In [148]:
# in the given data set it have only 150 rows actually it is small dataset due to this our team faced  many problems

## Challenge for us  
The main challenge was that the input features had a **complex and highly nonlinear relationship** with `overall_yield`, making it difficult for the model to capture the underlying pattern directly.

To improve this, we used **domain-inspired feature engineering** to create features that better represent the physical and kinetic relationships within the process.


In [3]:
df

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield
0,33.09,3.68,357.75,19.87,383.79,63.024
1,76.30,1.34,429.70,14.84,405.72,86.611
2,59.90,1.01,431.10,11.76,385.40,86.347
3,49.90,2.21,445.61,22.85,367.74,92.175
4,16.70,3.95,458.91,4.56,374.13,82.211
...,...,...,...,...,...,...
145,7.77,2.33,492.09,12.90,529.91,0.000
146,50.72,3.19,497.90,17.35,426.78,1.249
147,42.70,1.26,463.01,5.96,458.71,0.645
148,8.86,2.68,406.44,6.42,531.29,0.000


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

In [35]:
df

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield,delta_T,residence_time_proxy
0,33.09,3.68,357.75,19.87,383.79,63.024,26.04,0.600484
1,76.30,1.34,429.70,14.84,405.72,86.611,-23.98,0.194495
2,59.90,1.01,431.10,11.76,385.40,86.347,-45.70,0.196327
3,49.90,2.21,445.61,22.85,367.74,92.175,-77.87,0.457916
4,16.70,3.95,458.91,4.56,374.13,82.211,-84.78,0.273054
...,...,...,...,...,...,...,...,...
145,7.77,2.33,492.09,12.90,529.91,0.000,37.82,1.660232
146,50.72,3.19,497.90,17.35,426.78,1.249,-71.12,0.342074
147,42.70,1.26,463.01,5.96,458.71,0.645,-4.30,0.139578
148,8.86,2.68,406.44,6.42,531.29,0.000,124.85,0.724605


In [6]:
X = df.drop(columns="overall_yield")
y = df["overall_yield"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        
    ]
]

y = df["overall_yield"]


In [114]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [20.4153351  22.96436051 23.17754777 19.44646388 16.67848942]
Mean CV RMSE: 20.536439334373636
Std CV RMSE: 2.4065221923795437


# features

In [41]:
# temp diff
df["delta_T"] = (
    df["jacket_temperature_K"]
    - df["inlet_temperature_K"]
)


In [42]:
df

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield,delta_T,residence_time_proxy
0,33.09,3.68,357.75,19.87,383.79,63.024,26.04,0.600484
1,76.30,1.34,429.70,14.84,405.72,86.611,-23.98,0.194495
2,59.90,1.01,431.10,11.76,385.40,86.347,-45.70,0.196327
3,49.90,2.21,445.61,22.85,367.74,92.175,-77.87,0.457916
4,16.70,3.95,458.91,4.56,374.13,82.211,-84.78,0.273054
...,...,...,...,...,...,...,...,...
145,7.77,2.33,492.09,12.90,529.91,0.000,37.82,1.660232
146,50.72,3.19,497.90,17.35,426.78,1.249,-71.12,0.342074
147,42.70,1.26,463.01,5.96,458.71,0.645,-4.30,0.139578
148,8.86,2.68,406.44,6.42,531.29,0.000,124.85,0.724605


# model with timeproxy ftr

In [34]:
df["residence_time_proxy"] = (
    df["length_m"] /
    df["flow_rate_L_min"]
)

In [36]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy"
    ]
]

y = df["overall_yield"]


In [55]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [21.44444978 24.39507396 23.26145239 18.87234206 17.56842966]
Mean CV RMSE: 21.10834957124083
Std CV RMSE: 2.5722416229648264


In [ ]:
# model with time proxy and delta t

In [43]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy",
        "delta_T"
    ]
]

y = df["overall_yield"]


In [52]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [21.44444978 24.39507396 23.26145239 18.87234206 17.56842966]
Mean CV RMSE: 21.10834957124083
Std CV RMSE: 2.5722416229648286


In [ ]:
# + thermal bala

In [56]:
df["thermal_exposure"] = (
    df["delta_T"] *
    df["residence_time_proxy"]
)

In [58]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy",
        "delta_T",
        "thermal_exposure"
    ]
]

y = df["overall_yield"]


In [59]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [20.85876474 24.11062886 23.95632883 19.3757156  17.38592216]
Mean CV RMSE: 21.137472036545834
Std CV RMSE: 2.6092606257382136


In [ ]:
# + arhennius

In [60]:
df["inv_T"] = (
    1 / df["inlet_temperature_K"]
)

In [62]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy",
        "delta_T",
        "thermal_exposure",
        "inv_T"
    ]
]

y = df["overall_yield"]


In [65]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [20.89183992 23.56931859 23.91217153 19.26191943 16.76712615]
Mean CV RMSE: 20.88047512373242
Std CV RMSE: 2.68181109141085


In [ ]:
# + time curveture

In [72]:
df["residence_time_proxy_sq"] = (
    df["residence_time_proxy"] ** 2
)

In [73]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy",
        "delta_T",
        "thermal_exposure",
        "inv_T",
        "residence_time_proxy_sq"
    ]
]

y = df["overall_yield"]


In [70]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [21.02759037 23.44110059 23.31220896 19.40747639 16.37782834]
Mean CV RMSE: 20.71324092945607
Std CV RMSE: 2.6380103127250982


In [ ]:
# proxy ka square

In [71]:
df["residence_time_proxy_sq"] = (
    df["residence_time_proxy"] ** 2
)

In [ ]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        "residence_time_proxy",
        
        "thermal_exposure",
        "inv_T",
        "residence_time_proxy_sq"
    ]
]

y = df["overall_yield"]


In [75]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [21.02759037 23.44110059 23.31220896 19.40747639 16.37782834]
Mean CV RMSE: 20.71324092945607
Std CV RMSE: 2.638010312725099


In [76]:
print(df.columns.tolist())

['flow_rate_L_min', 'concentration_mol_L', 'inlet_temperature_K', 'length_m', 'jacket_temperature_K', 'overall_yield', 'delta_T', 'residence_time_proxy', 'thermal_exposure', 'inv_T', 'residence_time_proxy_sq']


In [ ]:
# + new feature

In [77]:
df["kinetic_exposure_proxy"] = (
    df["residence_time_proxy"] *
    df["inv_T"]
)

In [88]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        
        "residence_time_proxy",
       
        "thermal_exposure",
        "inv_T",
        
        "kinetic_exposure_proxy"
    ]
]

y = df["overall_yield"]


# best yhi hai abhi tak

In [99]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.8,
    bootstrap=True,
    max_samples=1.0,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [20.1441594  22.70028915 22.84992167 18.71638129 15.64491943]
Mean CV RMSE: 20.011134189856897
Std CV RMSE: 2.6850765667943723


# extra tree regressor

In [105]:
for leaf in [1, 2, 3, 4, 5]:

    etr = ExtraTreesRegressor(
        n_estimators=400,
        max_depth=10,
        min_samples_split=2,
        min_samples_leaf=leaf,
        max_features=0.8,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

    scores = -cross_val_score(
        etr,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    print(
        "Leaf:", leaf,
        "| Mean CV RMSE:", round(scores.mean(), 4),
        "| Std:", round(scores.std(), 4)
    )

Leaf: 1 | Mean CV RMSE: 18.0965 | Std: 2.4201
Leaf: 2 | Mean CV RMSE: 18.6373 | Std: 2.3433
Leaf: 3 | Mean CV RMSE: 20.1378 | Std: 2.255
Leaf: 4 | Mean CV RMSE: 21.298 | Std: 1.8045
Leaf: 5 | Mean CV RMSE: 22.2099 | Std: 1.5662


# gradient boosting

In [108]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42
)

scores = -cross_val_score(
    gbr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Fold RMSE:", scores)
print("Mean CV RMSE:", scores.mean())
print("Std CV RMSE:", scores.std())

Fold RMSE: [20.29161891 23.39732978 22.51243993 18.00573182 17.03048951]
Mean CV RMSE: 20.247521988540733
Std CV RMSE: 2.466901216863284


# hist gradient

In [112]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.0,
    random_state=42
)

scores = -cross_val_score(
    hgb,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Fold RMSE:", scores)
print("Mean CV RMSE:", scores.mean())
print("Std CV RMSE:", scores.std())

Fold RMSE: [19.75654525 26.24603056 25.81194979 22.95834627 19.93365414]
Mean CV RMSE: 22.94130520289973
Std CV RMSE: 2.769503971590507


In [ ]:
# ek aur naya aur purane bale

In [84]:
df["concentration_temp_proxy"] = (
    df["concentration_mol_L"] *
    df["inv_T"]
)

In [85]:
X = df[
    [
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",
        
        "residence_time_proxy",
       
        "thermal_exposure",
        "inv_T",
        "concentration_temp_proxy",
        "kinetic_exposure_proxy"
    ]
]

y = df["overall_yield"]


In [111]:
rfr = RandomForestRegressor(
    n_estimators=400,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.8,
    bootstrap=True,
    max_samples=0.9,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rfr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rmse = -scores

print("Fold RMSE:", rmse)
print("Mean CV RMSE:", rmse.mean())
print("Std CV RMSE:", rmse.std())

Fold RMSE: [20.4153351  22.96436051 23.17754777 19.44646388 16.67848942]
Mean CV RMSE: 20.536439334373636
Std CV RMSE: 2.406522192379544


In [74]:
df

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield,delta_T,residence_time_proxy,thermal_exposure,inv_T,residence_time_proxy_sq
0,33.09,3.68,357.75,19.87,383.79,63.024,26.04,0.600484,15.636591,0.002795,0.360580
1,76.30,1.34,429.70,14.84,405.72,86.611,-23.98,0.194495,-4.664000,0.002327,0.037828
2,59.90,1.01,431.10,11.76,385.40,86.347,-45.70,0.196327,-8.972154,0.002320,0.038544
3,49.90,2.21,445.61,22.85,367.74,92.175,-77.87,0.457916,-35.657906,0.002244,0.209687
4,16.70,3.95,458.91,4.56,374.13,82.211,-84.78,0.273054,-23.149509,0.002179,0.074558
...,...,...,...,...,...,...,...,...,...,...,...
145,7.77,2.33,492.09,12.90,529.91,0.000,37.82,1.660232,62.789961,0.002032,2.756369
146,50.72,3.19,497.90,17.35,426.78,1.249,-71.12,0.342074,-24.328312,0.002008,0.117015
147,42.70,1.26,463.01,5.96,458.71,0.645,-4.30,0.139578,-0.600187,0.002160,0.019482
148,8.86,2.68,406.44,6.42,531.29,0.000,124.85,0.724605,90.466930,0.002460,0.525052


# test the test data with our model

In [115]:
ml_test_data = pd.read_csv("test_dataset.csv")

In [116]:
ml_test_data

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K
0,43.73,2.88,454.18,7.66,440.61
1,47.80,2.33,401.37,19.95,390.13
2,7.14,0.65,411.86,3.05,476.23
3,17.86,1.28,385.97,22.36,353.56
4,56.40,2.51,495.33,8.36,517.71
5,67.54,0.92,393.28,15.36,455.11
6,28.02,2.25,372.10,22.95,424.19
7,72.02,0.98,369.40,24.83,435.15
8,59.12,0.68,491.61,3.43,403.02
9,19.25,1.12,402.72,23.36,439.28


In [117]:
# required features in the data set

In [123]:
ml_test_data["residence_time_proxy"] = (
    ml_test_data["length_m"] /
    ml_test_data["flow_rate_L_min"]
)

In [127]:
ml_test_data["delta_T"] = (
    ml_test_data["jacket_temperature_K"]
    - ml_test_data["inlet_temperature_K"]
)

In [128]:
ml_test_data["thermal_exposure"] = (
    ml_test_data["delta_T"] *
    ml_test_data["residence_time_proxy"]
)

In [129]:
ml_test_data["inv_T"] = (
    1 / ml_test_data["inlet_temperature_K"]
)

In [126]:
ml_test_data["kinetic_exposure_proxy"] = (
    ml_test_data["residence_time_proxy"] *
    ml_test_data["inv_T"]
)

In [131]:
X_ml_dataset = ml_test_data[
    [
        # feature already given in the data set
        "flow_rate_L_min",
        "concentration_mol_L",
        "inlet_temperature_K",
        "length_m",
        "jacket_temperature_K",

        # features discover
        "residence_time_proxy",
       
        "thermal_exposure",
        "inv_T",
        
        "kinetic_exposure_proxy"
    ]
]


In [130]:
ml_test_data

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,residence_time_proxy,inv_T,kinetic_exposure_proxy,delta_T,thermal_exposure
0,43.73,2.88,454.18,7.66,440.61,0.175166,0.002202,0.000386,-13.57,-2.377000
1,47.80,2.33,401.37,19.95,390.13,0.417364,0.002491,0.001040,-11.24,-4.691172
2,7.14,0.65,411.86,3.05,476.23,0.427171,0.002428,0.001037,64.37,27.496989
3,17.86,1.28,385.97,22.36,353.56,1.251960,0.002591,0.003244,-32.41,-40.576013
4,56.40,2.51,495.33,8.36,517.71,0.148227,0.002019,0.000299,22.38,3.317319
5,67.54,0.92,393.28,15.36,455.11,0.227421,0.002543,0.000578,61.83,14.061427
6,28.02,2.25,372.10,22.95,424.19,0.819058,0.002687,0.002201,52.09,42.664722
7,72.02,0.98,369.40,24.83,435.15,0.344765,0.002707,0.000933,65.75,22.668321
8,59.12,0.68,491.61,3.43,403.02,0.058018,0.002034,0.000118,-88.59,-5.139778
9,19.25,1.12,402.72,23.36,439.28,1.213506,0.002483,0.003013,36.56,44.365797


# Extra tree regressor

In [138]:
from sklearn.ensemble import ExtraTreesRegressor

etr = ExtraTreesRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.8,
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

scores = -cross_val_score(
    etr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Fold RMSE:", scores)
print("Mean CV RMSE:", scores.mean())
print("Std CV RMSE:", scores.std())

Fold RMSE: [18.09909127 18.2960692  22.21840821 14.7291003  17.13984409]
Mean CV RMSE: 18.096502614925633
Std CV RMSE: 2.4201046708060234


In [139]:
scores = -cross_val_score(
    etr,
    X,
    y,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Fold RMSE:", scores)
print("Mean CV RMSE:", scores.mean())
print("Std CV RMSE:", scores.std())

Fold RMSE: [18.09909127 18.2960692  22.21840821 14.7291003  17.13984409]
Mean CV RMSE: 18.096502614925633
Std CV RMSE: 2.4201046708060243


In [140]:
etr.fit(X, y)
y_pred_test_data_yield = etr.predict(X_ml_dataset)

In [141]:
print(pd.Series(y_pred_test_data_yield))

0     25.922762
1     78.410865
2     32.651526
3     82.729461
4      0.405869
5     56.343652
6     73.558277
7     68.761527
8      1.784382
9     47.333332
10     0.327978
11     0.282931
12    11.140154
13    52.801216
14     3.816934
15     2.503103
16     2.063199
17     2.764633
18     2.109961
19     1.559561
20    62.149577
21     3.991225
22     0.430535
23    86.161711
24    33.216631
25    25.455831
26    70.979338
27    42.208760
28    70.124694
29     4.010021
30    68.192506
31    63.673180
32    48.145907
33     0.773132
34    20.401762
35    17.736737
36    16.355917
37    55.405379
38     0.333391
39    57.746420
40     2.181356
41    29.031622
42     3.356935
43     4.253637
44    57.003085
45    75.773517
46     1.934371
47     1.397222
48    24.551039
49     2.329279
dtype: float64


In [142]:
submission = pd.DataFrame({
    "overall_yield": y_pred_test_data_yield
})

submission.to_csv("submission.csv", index=False)

display(submission)

,overall_yield
0,25.922762
1,78.410865
2,32.651526
3,82.729461
4,0.405869
5,56.343652
6,73.558277
7,68.761527
8,1.784382
9,47.333332
